In [3]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [6]:
llm_model = "gpt-4o-mini"

file = 'data/OutdoorClothingCatalog_1000.csv'

In [20]:
import csv
from langchain_core.documents import Document

docs = []
with open(file=file, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        content = "\n".join(f"{k}: {v}" for k, v in row.items())
        docs.append(Document(page_content=content, metadata={"row": i, "source": file}))

In [9]:
# from langchain.indexes import VectorstoreIndexCreator

# #pip install docarray

# index = VectorstoreIndexCreator(
#     vectorstore_cls=DocArrayInMemorySearch
# ).from_loaders([loader])

# query ="Please list all your shirts with sun protection \
# in a table in markdown and summarize each one."

# llm_replacement_model = OpenAI(temperature=0, 
#                                model='gpt-3.5-turbo-instruct')

# response = index.query(query, 
#                        llm = llm_replacement_model)

# display(Markdown(response))



In [40]:
#!pip install -U langchain-chroma langchain-openai langchain-core chromadb


from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from IPython.display import Markdown, display

In [38]:
#!pip install docarray langchain-community langchain-openai

#!pip uninstall -y openai langchain-openai
#!pip install -U openai langchain-openai

#!pip uninstall -y openai langchain-openai

#!pip cache purge

#!pip install --no-cache-dir -U openai langchain-openai

#!pip show openai

In [41]:
# 1. Construir el vectorstore directamente desde el loader
embeddings = OpenAIEmbeddings()

#vectorstore = DocArrayInMemorySearch.from_documents(docs, embeddings)
#retriever = vectorstore.as_retriever()

ImportError: cannot import name 'path_template' from 'openai._utils' (/Users/danielp/GitRepo/LangChainForLLM/.venv/lib/python3.10/site-packages/openai/_utils/__init__.py)

In [ ]:
# 2. Prompt explícito (antes esto vivía escondido dentro de index.query)
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}"""
)

In [ ]:
# 3. Modelo de chat (gpt-3.5-turbo-instruct es un modelo de completions legacy;
#    si quieres seguir usándolo tal cual, usa langchain_openai.OpenAI en vez de ChatOpenAI)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# 4. Cadena LCEL equivalente a index.query()
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

In [ ]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# 5. Ejecutar
query = (
    "Please list all your shirts with sun protection "
    "in a table in markdown and summarize each one."
)

In [ ]:
response = chain.invoke(query)
display(Markdown(response))